# VoxLingo — synchronisation labiale (LatentSync 1.5) sur le GPU gratuit de Kaggle

Ce carnet fait correspondre les lèvres de la vidéo d'origine à la voix doublée par VoxLingo.

**Avant de lancer :**
1. Dans le panneau de droite, *Session options* → *Accelerator* : choisissez **GPU T4 x2** ou **GPU P100**.
2. *Internet* : **activé** (pour télécharger LatentSync, une seule fois par session).
3. *Add Input* → *Upload* : importez le fichier **`voxlingo_kaggle_<projet>.zip`** téléchargé depuis VoxLingo (écran Résultat).

Puis *Run All*. Comptez environ 3 à 6 minutes de calcul par minute de vidéo. À la fin, téléchargez **`video_synchronisee.mp4`** (dossier *Output*) et importez-la dans VoxLingo.

> Usage personnel : ne synchronisez que des vidéos dont vous avez les droits, avec l'accord des personnes filmées.

In [ ]:
# 1. Retrouver le paquet VoxLingo importé (vidéo d'origine + piste doublée)
import glob, os, zipfile, shutil

TRAVAIL = '/kaggle/working/voxlingo'
os.makedirs(TRAVAIL, exist_ok=True)
zips = glob.glob('/kaggle/input/**/voxlingo_kaggle_*.zip', recursive=True)
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(TRAVAIL)
else:  # Kaggle décompresse parfois les .zip automatiquement
    for f in glob.glob('/kaggle/input/**/*', recursive=True):
        if os.path.basename(f) in ('video_originale.mp4', 'voix_doublee.wav'):
            shutil.copy(f, TRAVAIL)
VIDEO = os.path.join(TRAVAIL, 'video_originale.mp4')
AUDIO = os.path.join(TRAVAIL, 'voix_doublee.wav')
assert os.path.isfile(VIDEO) and os.path.isfile(AUDIO), "Paquet VoxLingo introuvable : ajoutez-le avec « Add Input → Upload »."
print('Vidéo :', VIDEO)
print('Audio :', AUDIO)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Installer LatentSync 1.5 (ByteDance) et ses poids
%cd /kaggle/working
!git clone --depth 1 https://github.com/bytedance/LatentSync.git
%cd /kaggle/working/LatentSync
!pip install -q -r requirements.txt
!pip install -q huggingface_hub
!huggingface-cli download ByteDance/LatentSync-1.5 whisper/tiny.pt --local-dir checkpoints
!huggingface-cli download ByteDance/LatentSync-1.5 latentsync_unet.pt --local-dir checkpoints
!ls -lh checkpoints checkpoints/whisper

In [ ]:
# 3. Synchroniser les lèvres sur la voix doublée
#    inference_steps : 20 (qualité) ; baissez à 10 pour aller plus vite.
SORTIE = '/kaggle/working/video_synchronisee.mp4'
!python -m scripts.inference \
    --unet_config_path configs/unet/stage2.yaml \
    --inference_ckpt_path checkpoints/latentsync_unet.pt \
    --inference_steps 20 \
    --guidance_scale 1.5 \
    --video_path "{VIDEO}" \
    --audio_path "{AUDIO}" \
    --video_out_path "{SORTIE}"
print('Terminé :' if os.path.isfile(SORTIE) else 'Échec : relisez les messages ci-dessus.', SORTIE)

## 4. Récupérer le résultat

Dans le panneau de droite, *Output* → `video_synchronisee.mp4` → *Download*.

Dans VoxLingo, écran **Résultat** du projet → *Synchronisation labiale* → **Importer la vidéo synchronisée**.

**En cas de problème**
- *CUDA out of memory* : relancez la cellule 3 avec `--inference_steps 10`, ou découpez la vidéo.
- *No face detected* : LatentSync a besoin d'un visage visible de face ; les plans sans visage gardent la vidéo d'origine.
- Le dépôt LatentSync évolue : si une commande échoue, consultez https://github.com/bytedance/LatentSync (section *Inference*).